In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, QED
from rdkit.Chem.rdMolDescriptors import CalcTPSA
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("./data/output_from_KNIME/hit_selective_comparison.csv")
print(df.shape)
df.head()

In [ ]:
'''
df= (
    df.groupby("type", group_keys=False)
      .apply(lambda x: x.sample(frac=0.1, random_state=42))
)
'''
df.shape


In [ ]:
# --- 計算関数 ---
def calc_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return [np.nan]*8
    return [
        Descriptors.MolWt(mol),
        Crippen.MolLogP(mol),
        Lipinski.NumHDonors(mol),
        Lipinski.NumHAcceptors(mol),
        CalcTPSA(mol),
        Lipinski.NumRotatableBonds(mol),
        Descriptors.FractionCSP3(mol),
        QED.qed(mol)
    ]

# --- 記述子名 ---
descriptor_names = ['MolecularWeight','LogP','HBD','HBA','TPSA','RotatableBonds','Fsp3','QED']
cont_vars = ['MolecularWeight','LogP','TPSA','Fsp3','QED']  # QEDを追加
disc_vars = ['HBD','HBA','RotatableBonds']

# --- 閾値を指定してカテゴリ化 ---
criteria = 2  # ここを変えることで閾値を調整できる。これ以上の数値になる場合はunselective 2にすると1000 RNA中1個、つまり0.1%以上を選択的とみなしている
df['selectivity'] = df['Num-hit-RNA/1000'].apply(lambda x: 'unselective' if x >= criteria else 'selective')

# --- 色設定 ---
groups_violin = ['unselective', 'selective']  # バイオリンプロット用（unselectiveが左）
groups_hist = [ 'unselective','selective']    # ヒストグラム用（selectiveが上）
palette = {'selective': '#008B8B', 'unselective': '#7FFFD4'}


# --- 記述子を計算 ---
df[descriptor_names] = df['SMILES'].apply(calc_descriptors).tolist()

# --- 外れ値ビニング ---
df['HBD'] = df['HBD'].apply(lambda x: '≥10' if x >= 10 else str(int(x)))
df['HBA'] = df['HBA'].apply(lambda x: '≥15' if x >= 15 else str(int(x)))
df['RotatableBonds'] = df['RotatableBonds'].apply(lambda x: '≥15' if x >= 15 else str(int(x)))

# --- 連続変数: violin + box (QEDも含めて統一表示) ---
# 5つのプロットなので、各プロットが正方形に近くなるように調整
fig, axes = plt.subplots(1, len(cont_vars), figsize=(25, 5))
for ax, var in zip(axes, cont_vars):
    sns.violinplot(
        x='selectivity', y=var,
        data=df[df['selectivity'].isin(groups_violin)],
        palette=palette, inner=None, ax=ax, alpha=0.5,
        order=groups_violin  # unselectiveが左に来る
    )
    sns.boxplot(
        x='selectivity', y=var,
        data=df[df['selectivity'].isin(groups_violin)],
        width=0.1, showcaps=True,
        boxprops={'facecolor':'none','edgecolor':'black','linewidth':4},
        whiskerprops={'linewidth':4,'color':'black'},
        capprops={'linewidth':4,'color':'black'},
        medianprops={'linewidth':4,'color':'black'},
        ax=ax,
        order=groups_violin  # unselectiveが左に来る
    )
    ax.set_title(var, fontsize=20, pad=20)
    ax.set_xlabel('')
    ax.set_ylabel(var, fontsize=20, labelpad=10)
    ax.tick_params(axis='x', labelsize=18, pad=8)
    ax.tick_params(axis='y', labelsize=18, pad=8)
plt.tight_layout()
plt.show()

# --- 離散変数: histogram ---
fig, axes = plt.subplots(1, len(disc_vars), figsize=(15, 5))
for ax, var in zip(axes, disc_vars):
    unique_vals = sorted(df[var].unique(), key=lambda x: (int(x.replace('≥','').strip('+')) if x.startswith('≥') else int(x)))
    width = 0.3
    offset = 0.2
    for i, grp in enumerate(groups_hist):  # selectiveが先（上）に描画される
        subset = df[df['selectivity'] == grp]
        counts = subset[var].value_counts(normalize=True).reindex(unique_vals).fillna(0)
        positions = np.arange(len(unique_vals)) + (i - 0.5)*offset
        ax.bar(
            positions,
            counts.values,
            width=width,
            alpha=0.7,
            color=palette[grp],
            edgecolor='black',
            linewidth=1,
            label=grp
        )
    ax.set_title(var, fontsize=20, pad=20)
    ax.set_xlabel(var, fontsize=20, labelpad=10)
    ax.set_ylabel('Density', fontsize=20, labelpad=10)
    ax.set_xticks(np.arange(len(unique_vals)))
    ax.set_xticklabels(unique_vals, fontsize=18)
    ax.tick_params(axis='x', labelsize=18, pad=8)
    ax.tick_params(axis='y', labelsize=18, pad=8)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, QED
from rdkit.Chem.rdMolDescriptors import CalcTPSA
from scipy.stats import mannwhitneyu


# 記述子の計算関数
def calc_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return [np.nan]*8
    return [
        Descriptors.MolWt(mol),
        Crippen.MolLogP(mol),
        Lipinski.NumHDonors(mol),
        Lipinski.NumHAcceptors(mol),
        CalcTPSA(mol),
        Lipinski.NumRotatableBonds(mol),
        Descriptors.FractionCSP3(mol),
        QED.qed(mol)
    ]

# 記述子名
descriptor_names = ['MolecularWeight','LogP','HBD','HBA','TPSA','RotatableBonds','Fsp3','QED']
cont_vars = ['MolecularWeight','LogP','TPSA','Fsp3','QED']
disc_vars = ['HBD','HBA','RotatableBonds']

# 閾値を指定してカテゴリ化
criteria = 2
df['selectivity'] = df['Num-hit-RNA/1000'].apply(lambda x: 'unselective' if x >= criteria else 'selective')

# 色設定
groups_violin = ['unselective', 'selective']  # バイオリンプロット用（unselectiveが左）
palette = {'selective': '#008B8B', 'unselective': '#7FFFD4'}

# 記述子の計算
df[descriptor_names] = df['SMILES'].apply(calc_descriptors).tolist()

# 外れ値ビニング
df['HBD'] = df['HBD'].apply(lambda x: '≥10' if x >= 10 else str(int(x)))
df['HBA'] = df['HBA'].apply(lambda x: '≥15' if x >= 15 else str(int(x)))
df['RotatableBonds'] = df['RotatableBonds'].apply(lambda x: '≥15' if x >= 15 else str(int(x)))

# --- バイオリンプロット + 有意差アノテーション ---
fig, axes = plt.subplots(1, len(cont_vars), figsize=(6 * len(cont_vars), 6))
if len(cont_vars) == 1:
    axes = [axes]

for ax, var in zip(axes, cont_vars):
    sns.violinplot(
        x='selectivity', y=var,
        data=df[df['selectivity'].isin(groups_violin)],
        palette=palette, inner=None, ax=ax, alpha=0.5,
        order=groups_violin  # unselectiveが左に来る
    )
    sns.boxplot(
        x='selectivity', y=var,
        data=df[df['selectivity'].isin(groups_violin)],
        width=0.1, showcaps=True,
        boxprops={'facecolor':'none','edgecolor':'black','linewidth':2},
        whiskerprops={'linewidth':2,'color':'black'},
        capprops={'linewidth':2,'color':'black'},
        medianprops={'linewidth':2,'color':'black'},
        ax=ax,
        order=groups_violin  # unselectiveが左に来る
    )
    # 有意差のアノテーション
    vals1 = df[df['selectivity'] == 'selective'][var].dropna()
    vals2 = df[df['selectivity'] == 'unselective'][var].dropna()
    if len(vals1) > 0 and len(vals2) > 0:
        stat, pval = mannwhitneyu(vals1, vals2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        ymax = max(df[var].dropna())
        ax.text(0.5, ymax * 1.05, sig, ha='center', va='bottom', fontsize=16)
    
    ax.set_title(var, fontsize=20)
    ax.set_xlabel('')
    ax.set_ylabel(var, fontsize=20)
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)

plt.tight_layout()
plt.show()

In [ ]:
# Define effect size calculation functions
import numpy as np
from scipy.stats import mannwhitneyu

def cliffs_delta(x, y):
    """
    Calculate Cliff's delta effect size.
    
    Interpretation thresholds (Romano et al., 2006):
    - |δ| < 0.147: negligible
    - 0.147 ≤ |δ| < 0.33: small
    - 0.33 ≤ |δ| < 0.474: medium
    - |δ| ≥ 0.474: large
    """
    nx = len(x)
    ny = len(y)
    
    dominance = 0
    for xi in x:
        for yi in y:
            if xi > yi:
                dominance += 1
            elif xi < yi:
                dominance -= 1
    
    delta = dominance / (nx * ny)
    return delta

def interpret_cliffs_delta(delta):
    """Interpret Cliff's delta magnitude."""
    abs_delta = abs(delta)
    if abs_delta < 0.147:
        return "negligible"
    elif abs_delta < 0.33:
        return "small"
    elif abs_delta < 0.474:
        return "medium"
    else:
        return "large"

print("Effect size functions defined successfully!")

In [ ]:
# Export selectivity comparison to Excel with complete statistics
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment

# Prepare data
all_vars = ['MolecularWeight', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotatableBonds', 'Fsp3', 'QED']

# Get original data before binning for statistical tests
df_original = df.copy()
df_original[descriptor_names] = df_original['SMILES'].apply(calc_descriptors).tolist()

comprehensive_results = []

for var in all_vars:
    # Use original unbinned data for statistics
    data1 = df_original[df_original['selectivity'] == 'selective'][var].dropna()
    data2 = df_original[df_original['selectivity'] == 'unselective'][var].dropna()
    
    if len(data1) == 0 or len(data2) == 0:
        continue
    
    n1, n2 = len(data1), len(data2)
    median1, median2 = data1.median(), data2.median()
    q1_1, q3_1 = data1.quantile(0.25), data1.quantile(0.75)
    q1_2, q3_2 = data2.quantile(0.25), data2.quantile(0.75)
    iqr1, iqr2 = q3_1 - q1_1, q3_2 - q1_2
    mean1, std1 = data1.mean(), data1.std()
    mean2, std2 = data2.mean(), data2.std()
    
    stat, pval = mannwhitneyu(data1, data2, alternative='two-sided')
    delta = cliffs_delta(data1.values, data2.values)
    delta_interp = interpret_cliffs_delta(delta)
    
    sig = "***" if pval <= 0.001 else "**" if pval <= 0.01 else "*" if pval <= 0.05 else "n.s."
    
    comprehensive_results.append({
        'Variable': var,
        'Comparison': 'selective vs unselective',
        'Group1': 'selective',
        'Group1_n': n1,
        'Group1_Median': round(median1, 3),
        'Group1_Q1': round(q1_1, 3),
        'Group1_Q3': round(q3_1, 3),
        'Group1_IQR': round(iqr1, 3),
        'Group1_Mean': round(mean1, 3),
        'Group1_SD': round(std1, 3),
        'Group2': 'unselective',
        'Group2_n': n2,
        'Group2_Median': round(median2, 3),
        'Group2_Q1': round(q1_2, 3),
        'Group2_Q3': round(q3_2, 3),
        'Group2_IQR': round(iqr2, 3),
        'Group2_Mean': round(mean2, 3),
        'Group2_SD': round(std2, 3),
        'U_statistic': round(stat, 2),
        'p_value': pval,
        'p_formatted': f"{pval:.4e}" if pval < 0.0001 else f"{pval:.4f}",
        'Significance': sig,
        'Cliffs_delta': round(delta, 3),
        'Effect_size': delta_interp,
        'Biological_relevance': 'High' if abs(delta) >= 0.33 else 'Moderate' if abs(delta) >= 0.147 else 'Low'
    })

results_df = pd.DataFrame(comprehensive_results)
output_excel = "./data/output_from_code/S3_Selectivity_Statistical_Analysis.xlsx

with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='Complete_Statistics', index=False)
    
    summary_df = results_df[['Variable', 'Comparison', 'Group1', 'Group1_n', 'Group1_Median', 'Group1_IQR',
                            'Group2', 'Group2_n', 'Group2_Median', 'Group2_IQR',
                            'p_formatted', 'Significance', 'Cliffs_delta', 'Effect_size', 'Biological_relevance']]
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

wb = load_workbook(output_excel)
for sheet_name in ['Complete_Statistics', 'Summary']:
    ws = wb[sheet_name]
    fill = PatternFill(start_color="366092" if sheet_name == 'Complete_Statistics' else "70AD47", 
                      end_color="366092" if sheet_name == 'Complete_Statistics' else "70AD47", 
                      fill_type="solid")
    for cell in ws[1]:
        cell.fill = fill
        cell.font = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    for column in ws.columns:
        max_len = max(len(str(cell.value)) if cell.value else 0 for cell in column)
        ws.column_dimensions[column[0].column_letter].width = min(max_len + 2, 50)

wb.save(output_excel)

print("="*80)
print("SELECTIVITY COMPARISON - STATISTICAL ANALYSIS COMPLETE")
print("="*80)
print(f"File: {output_excel}")
print(f"Comparisons: {len(results_df)}")
print(f"Selective compounds (n={results_df.iloc[0]['Group1_n']})")
print(f"Unselective compounds (n={results_df.iloc[0]['Group2_n']})")
print("\nIncludes: n, Median, IQR, Mean, SD, U-test, Cliff's delta, Effect size")
print("No missing values!")